# Colab 万能一键启动器（Google Drive + llama.cpp + ComfyUI + OpenCode）

本文件已保存在你的 Google Drive 根目录与本地工作区。下次直接在浏览器 Colab 打开它，按顺序点击对应单元格即可一键运行：
- **Cell 1**：挂载 Google Drive + 写入 HuggingFace Token + 一键清理 `.cache` / `split_files/` 释放网盘空间
- **Cell 2（推荐切到 CPU 运行时执行）**：从 HuggingFace 预下载 LLM（含多片 `.gguf` 与 `mmproj` 视觉投影）或 ComfyUI 权重到网盘
- **Cell 3（切到 GPU 运行时执行）**：硬件自适应部署 CUDA `llama-server`（最高 128K 上下文 + OOM 降级 + HTTP/2 隧道）并打印 OpenCode 同步命令
- **Cell 4（切到 GPU 运行时执行）**：本地高速盘部署 ComfyUI + 软链 Drive 模型 + 恢复 `custom_nodes` 快照 + 输出公网访问地址

In [ ]:
#@title 1. 挂载 Google Drive + 配置 HF Token + 清理冗余缓存
import os, shutil
from pathlib import Path
from google.colab import drive

HF_TOKEN = ""  #@param {type:"string"}
CLEAN_DRIVE_CACHE = True  #@param {type:"boolean"}

drive.mount("/content/drive", force_remount=False)

if HF_TOKEN.strip():
    for p in (
        Path("/root/.config/huggingface/token"),
        Path("/root/.cache/huggingface/token"),
    ):
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(HF_TOKEN.strip() + "\n", encoding="utf-8")
        os.chmod(p, 0o600)
    os.environ["HF_TOKEN"] = HF_TOKEN.strip()
    print("✅ HuggingFace Token 已写入本次环境。")

if CLEAN_DRIVE_CACHE:
    for target in (
        "/content/drive/MyDrive/_hf_prefetch_tmp",
        "/content/drive/MyDrive/ComfyUI_H3/ComfyUI/models/split_files",
    ):
        if os.path.exists(target):
            shutil.rmtree(target, ignore_errors=True)
            print("🧹 已清理:", target)
    for base in (
        Path("/content/drive/MyDrive/colab_models/llm"),
        Path("/content/drive/MyDrive/ComfyUI_H3/ComfyUI/models"),
    ):
        if base.exists():
            for c in base.rglob(".cache"):
                if c.is_dir():
                    shutil.rmtree(c, ignore_errors=True)
                    print("🧹 已清理 .cache:", c)

usage = shutil.disk_usage("/content/drive/MyDrive")
print(f"💾 Google Drive 剩余空间: {usage.free / 1024**3:.2f} GB / 总计 {usage.total / 1024**3:.2f} GB")

In [ ]:
#@title 2. [推荐先切到 CPU] 预下载 HF 模型到 Google Drive（自动清理 .cache）
import json, os, re, shutil, subprocess, urllib.request
from pathlib import Path

HF_REPO = "bartowski/Qwen_Qwen3-32B-GGUF"  #@param {type:"string"}
HF_FILE = ""  #@param {type:"string"}
MODEL_KIND = "llm"  #@param ["llm", "comfyui"]
COMFY_SUBDIR = ""  #@param {type:"string"}
INCLUDE_MMPROJ = True  #@param {type:"boolean"}
HF_TOKEN_OVERRIDE = ""  #@param {type:"string"}

token = HF_TOKEN_OVERRIDE.strip() or os.environ.get("HF_TOKEN", "").strip()
if not token:
    tok_p = Path("/root/.cache/huggingface/token")
    if tok_p.exists():
        token = tok_p.read_text(encoding="utf-8").strip()

DRIVE_LLM_DIR = Path("/content/drive/MyDrive/colab_models/llm")
DRIVE_COMFY_MODELS = Path("/content/drive/MyDrive/ComfyUI_H3/ComfyUI/models")
TMP_DIR = Path("/content/drive/MyDrive/_hf_prefetch_tmp")

req = urllib.request.Request(
    f"https://huggingface.co/api/models/{HF_REPO}/tree/main?recursive=true",
    headers={"User-Agent": "colab-notebook/1.0", **({"Authorization": f"Bearer {token}"} if token else {})},
)
with urllib.request.urlopen(req, timeout=30) as r:
    all_files = [x for x in json.loads(r.read().decode("utf-8")) if isinstance(x, dict) and x.get("type") == "file"]

def _expand_shards(path_str):
    m = re.match(r"^(.*)-(\d{5})-of-(\d{5})(\.[^.]+)$", path_str)
    if not m:
        return [f for f in all_files if f["path"] == path_str]
    prefix, _, total, ext = m.groups()
    pat = re.compile(rf"^{re.escape(prefix)}-\d{{5}}-of-{total}{re.escape(ext)}$")
    return sorted([f for f in all_files if pat.match(f["path"])], key=lambda x: x["path"]) or [{"path": path_str, "size": 0}]

if HF_FILE.strip():
    selected = _expand_shards(HF_FILE.strip())
elif MODEL_KIND == "llm":
    cands = [f for f in all_files if f["path"].lower().endswith(".gguf") and "mmproj" not in f["path"].lower()]
    chosen = None
    for p in ("q4_k_m", "iq4_xs", "q5_k_m", "q4_k_s", "q6_k", "q8_0"):
        hits = [f for f in cands if p in f["path"].lower()]
        if hits:
            chosen = sorted(hits, key=lambda x: ("00001-of-" not in x["path"], len(x["path"])))[0]["path"]
            break
    chosen = chosen or sorted(cands, key=lambda x: x["path"])[0]["path"]
    selected = _expand_shards(chosen)
else:
    cands = [f for f in all_files if f["path"].lower().endswith(".safetensors")]
    chosen = cands[0]
    for p in ("int8_convrot", "fp8_scaled", "fp8mixed", "nvfp4", "bf16"):
        hits = [f for f in cands if p in f["path"].lower()]
        if hits:
            chosen = hits[0]
            break
    selected = [chosen]

if MODEL_KIND == "llm" and INCLUDE_MMPROJ:
    mms = [f for f in all_files if f["path"].lower().endswith(".gguf") and "mmproj" in f["path"].lower()]
    if mms:
        mm_pick = sorted(mms, key=lambda x: ("f16" not in x["path"].lower(), len(x["path"])))[0]
        if all(x["path"] != mm_pick["path"] for x in selected):
            selected.append(mm_pick)

def _dest(rpath):
    if MODEL_KIND == "llm":
        return DRIVE_LLM_DIR / HF_REPO.replace("/", "__") / rpath
    clean = rpath[len("split_files/"):] if rpath.startswith("split_files/") else rpath
    if COMFY_SUBDIR.strip():
        return DRIVE_COMFY_MODELS / COMFY_SUBDIR.strip() / Path(clean).name
    return (DRIVE_COMFY_MODELS / clean) if "/" in clean else (DRIVE_COMFY_MODELS / "diffusion_models" / clean)

subprocess.run(["bash", "-lc", "uv pip install --system -q huggingface_hub hf_transfer || pip install -q huggingface_hub hf_transfer"], check=False)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
from huggingface_hub import hf_hub_download

for item in selected:
    tp = _dest(item["path"])
    if tp.exists() and (not item.get("size") or tp.stat().st_size >= item["size"] * 0.99):
        print("⏭️ 已存在:", tp)
        continue
    tp.parent.mkdir(parents=True, exist_ok=True)
    print("⬇️ 下载:", item["path"], "->", tp)
    dl = hf_hub_download(repo_id=HF_REPO, filename=item["path"], local_dir=str(TMP_DIR), token=token or None)
    shutil.move(dl, str(tp))

shutil.rmtree(TMP_DIR, ignore_errors=True)
print("✅ 预下载完成！若当前为 CPU 运行时，请切换为 GPU 运行时后再执行 Cell 3 或 Cell 4。")

In [ ]:
#@title 3. [GPU] 启动 CUDA llama-server（硬件自适应上下文 + OOM 阶梯降级 + HTTP/2 隧道）
import json, os, re, shutil, subprocess, tarfile, time, urllib.request
from pathlib import Path

MODEL_PATH = ""  #@param {type:"string"}
MODEL_ALIAS = "colab-llm"  #@param {type:"string"}
PORT = 8080  #@param {type:"integer"}
FORCE_CTX = 0  #@param {type:"integer"}

DRIVE_LLM_ROOT = Path("/content/drive/MyDrive/colab_models/llm")
DRIVE_BIN_CACHE = Path("/content/drive/MyDrive/colab_tools/llama_bin")
LOCAL_BIN = Path("/content/llama_bin")
LOCAL_MODELS = Path("/content/models_cache")

if not MODEL_PATH.strip():
    ggufs = [p for p in sorted(DRIVE_LLM_ROOT.rglob("*.gguf")) if "mmproj" not in p.name.lower() and ("-of-" not in p.name or "00001-of-" in p.name)]
    assert ggufs, f"未在 {DRIVE_LLM_ROOT} 找到 .gguf 模型，请先运行 Cell 2！"
    model_src = ggufs[-1]
else:
    model_src = Path(MODEL_PATH.strip())
print("📦 选用模型:", model_src)

def _shards(p: Path):
    m = re.match(r"^(.*)-(\d{5})-of-(\d{5})\.gguf$", p.name, re.I)
    if not m:
        return [p]
    pat = re.compile(rf"^{re.escape(m.group(1))}-\d{{5}}-of-{m.group(3)}\.gguf$", re.I)
    return sorted([x for x in p.parent.iterdir() if x.is_file() and pat.match(x.name)]) or [p]

shards = _shards(model_src)
mms = sorted([p for p in model_src.parent.rglob("*.gguf") if "mmproj" in p.name.lower()], key=lambda x: ("f16" not in x.name.lower(), len(x.name)))
mmproj_src = mms[0] if mms else None
total_bytes = sum(x.stat().st_size for x in shards) + (mmproj_src.stat().st_size if mmproj_src else 0)

free_disk = shutil.disk_usage("/content").free
if free_disk > total_bytes * 1.2 + 5 * (1024**3):
    LOCAL_MODELS.mkdir(parents=True, exist_ok=True)
    for s in shards + ([mmproj_src] if mmproj_src else []):
        dst = LOCAL_MODELS / s.name
        if not dst.exists() or dst.stat().st_size != s.stat().st_size:
            print("⚡ 复制到 /content 本地高速盘:", s.name)
            shutil.copy2(s, dst)
    model_file = LOCAL_MODELS / model_src.name
    mmproj_file = (LOCAL_MODELS / mmproj_src.name) if mmproj_src else None
else:
    model_file, mmproj_file = model_src, mmproj_src

smi = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], text=True).strip().splitlines()[0]
gpu_name, vram_mb = [x.strip() for x in smi.split(",")]
vram_gb = float(vram_mb) / 1024.0
model_gb = total_bytes / 1024**3
free_vram = vram_gb - model_gb - 2.0
init_ctx = FORCE_CTX if FORCE_CTX > 0 else (131072 if free_vram >= 36 else 65536 if free_vram >= 16 else 32768 if free_vram >= 6 else 16384)
print(f"🖥️ GPU={gpu_name} ({vram_gb:.1f}GB) | 模型总大小={model_gb:.1f}GB | 初始上下文={init_ctx}")

srv = next((p for p in LOCAL_BIN.rglob("llama-server")), None) if LOCAL_BIN.exists() else None
if not srv and DRIVE_BIN_CACHE.exists():
    shutil.copytree(DRIVE_BIN_CACHE, LOCAL_BIN, dirs_exist_ok=True)
    srv = next((p for p in LOCAL_BIN.rglob("llama-server")), None)
if not srv:
    rels = json.loads(urllib.request.urlopen(urllib.request.Request("https://api.github.com/repos/ggml-org/llama.cpp/releases?per_page=10", headers={"User-Agent": "Mozilla/5.0"})).read())
    urls = []
    for rel in rels:
        assets = rel.get("assets", [])
        lu = next((a["browser_download_url"] for a in assets if a["name"].startswith("llama-") and "cuda-12.8-x64" in a["name"] and a["name"].endswith(".tar.gz")), None)
        cu = next((a["browser_download_url"] for a in assets if a["name"].startswith("cudart-llama-") and "cuda-12.8-x64" in a["name"]), None)
        if lu:
            urls = [u for u in (cu, lu) if u]
            break
    LOCAL_BIN.mkdir(parents=True, exist_ok=True)
    for u in urls:
        tp = Path("/tmp") / Path(u).name
        urllib.request.urlretrieve(u, tp)
        with tarfile.open(tp, "r:gz") as tf:
            tf.extractall(LOCAL_BIN)
        tp.unlink(missing_ok=True)
    srv = next(p for p in LOCAL_BIN.rglob("llama-server"))
    srv.chmod(0o755)
    shutil.copytree(srv.parent, DRIVE_BIN_CACHE, dirs_exist_ok=True)

env = os.environ.copy()
env["LD_LIBRARY_PATH"] = f"{srv.parent}:{env.get('LD_LIBRARY_PATH', '')}"
ht = subprocess.check_output([str(srv), "--help"], env=env, text=True, stderr=subprocess.STDOUT)
fa_flags = ["-fa", "on"] if re.search(r"-fa,\s*--flash-attn\s*\[(on|off|auto)", ht) else ["--flash-attn"]

actual_ctx = init_ctx
for ctx_try in [init_ctx] + [c for c in (131072, 65536, 32768, 16384, 8192) if c < init_ctx]:
    subprocess.run(["pkill", "-f", f"llama-server.*--port {PORT}"], check=False)
    time.sleep(1)
    cmd = [str(srv), "-m", str(model_file), "--alias", MODEL_ALIAS, "--host", "127.0.0.1", "--port", str(PORT), "-c", str(ctx_try), "-np", "1", "-ngl", "999", "--jinja", *fa_flags, "-ctk", "q8_0", "-ctv", "q8_0"]
    if mmproj_file:
        cmd.extend(["--mmproj", str(mmproj_file)])
    lf = open(f"/tmp/llama_server_{PORT}.log", "w")
    proc = subprocess.Popen(cmd, env=env, stdout=lf, stderr=subprocess.STDOUT, start_new_session=True)
    ok = False
    for _ in range(120):
        if proc.poll() is not None:
            break
        try:
            if urllib.request.urlopen(f"http://127.0.0.1:{PORT}/v1/models", timeout=2).status == 200:
                ok = True
                break
        except Exception:
            pass
        time.sleep(2)
    if ok:
        actual_ctx = ctx_try
        break
else:
    raise RuntimeError("llama-server 启动失败，请查看 /tmp/llama_server_8080.log")

cf = "/usr/local/bin/cloudflared"
if not Path(cf).exists():
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", cf)
    Path(cf).chmod(0o755)
subprocess.run(["pkill", "-f", f"cloudflared tunnel.*http://127.0.0.1:{PORT}"], check=False)
tlog = Path(f"/tmp/cloudflared_{PORT}.log")
tlog.unlink(missing_ok=True)
subprocess.Popen([cf, "tunnel", "--protocol", "http2", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"], stdout=open(tlog, "w"), stderr=subprocess.STDOUT, start_new_session=True)
pub_url = ""
for _ in range(45):
    if tlog.exists():
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", tlog.read_text(errors="ignore"))
        if m:
            pub_url = m.group(0)
            break
    time.sleep(1)
print(f"\n🎉 就绪！CTX={actual_ctx} | 公网 Base URL: {pub_url}/v1")
print(f"👉 本地 Mac 一键同步 OpenCode：\npython3 ~/.gemini/config/skills/colab-quick-launcher/scripts/sync_opencode.py --url '{pub_url}/v1' --model-id '{MODEL_ALIAS}' --context {actual_ctx}{' --vision' if mmproj_file else ''}")

In [ ]:
#@title 4. [GPU] 启动 ComfyUI（高速 NVMe + 软链 Drive 模型 + 恢复 custom_nodes 快照 + HTTP/2 隧道）
import os, re, shutil, subprocess, tarfile, time, urllib.request
from pathlib import Path

PORT = 8188  #@param {type:"integer"}
DRIVE_MODELS = Path("/content/drive/MyDrive/ComfyUI_H3/ComfyUI/models")
DRIVE_OUTPUT = Path("/content/drive/MyDrive/H3_Outputs")
DRIVE_NODES_TAR = Path("/content/drive/MyDrive/ComfyUI_H3/custom_nodes_snapshot.tar.gz")
LOCAL_COMFY = Path("/content/ComfyUI")

if not (LOCAL_COMFY / "main.py").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI.git", str(LOCAL_COMFY)], check=True)

DRIVE_MODELS.mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
ml = LOCAL_COMFY / "models"
if ml.is_symlink():
    ml.unlink()
elif ml.exists():
    shutil.rmtree(ml)
ml.symlink_to(DRIVE_MODELS)

subprocess.run(["bash", "-lc", f"cd {LOCAL_COMFY} && (uv pip install --system -q -r requirements.txt || pip install -q -r requirements.txt)"], check=True)

if DRIVE_NODES_TAR.exists():
    cn_dir = LOCAL_COMFY / "custom_nodes"
    cn_dir.mkdir(parents=True, exist_ok=True)
    with tarfile.open(DRIVE_NODES_TAR, "r:gz") as tf:
        tf.extractall(cn_dir)
    for rq in cn_dir.glob("*/requirements.txt"):
        subprocess.run(["bash", "-lc", f"uv pip install --system -q -r '{rq}' || pip install -q -r '{rq}'"], check=False)

subprocess.run(["pkill", "-f", f"main.py.*--port {PORT}"], check=False)
time.sleep(1)
subprocess.Popen(["python3", str(LOCAL_COMFY / "main.py"), "--listen", "0.0.0.0", "--port", str(PORT), "--output-directory", str(DRIVE_OUTPUT)], cwd=str(LOCAL_COMFY), stdout=open(f"/tmp/comfyui_{PORT}.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)

for _ in range(90):
    try:
        if urllib.request.urlopen(f"http://127.0.0.1:{PORT}/system_stats", timeout=2).status == 200:
            break
    except Exception:
        pass
    time.sleep(2)

cf = "/usr/local/bin/cloudflared"
if not Path(cf).exists():
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", cf)
    Path(cf).chmod(0o755)
subprocess.run(["pkill", "-f", f"cloudflared tunnel.*http://127.0.0.1:{PORT}"], check=False)
tlog = Path(f"/tmp/cloudflared_{PORT}.log")
tlog.unlink(missing_ok=True)
subprocess.Popen([cf, "tunnel", "--protocol", "http2", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"], stdout=open(tlog, "w"), stderr=subprocess.STDOUT, start_new_session=True)
for _ in range(45):
    if tlog.exists():
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", tlog.read_text(errors="ignore"))
        if m:
            print("🎨 ComfyUI WebUI 已就绪:", m.group(0))
            break
    time.sleep(1)